# Next-word predictor with an LSTM
Text: Grimms' Fairy Tales (Project Gutenberg eBook #2591), one story per line in data/grimm.txt.
Every 10th story (from the 9th) is the validation set, used to stop training; every 10th story (from the 10th)
is the test set, used only once at the end. The rest train the model.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import re
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
try:
    tf.config.experimental.enable_op_determinism()   # same numbers on every run
except Exception as e:
    print("op determinism not available:", e)
DATA = Path("data")
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. The text, sentence by sentence

In [ ]:
stories = DATA.joinpath("grimm.txt").read_text().strip().split("\n")
part = np.arange(len(stories)) % 10                     # 0-7 train, 8 validation, 9 test
train_stories = [f for f, p in zip(stories, part) if p < 8]
val_stories = [f for f, p in zip(stories, part) if p == 8]
test_stories = [f for f, p in zip(stories, part) if p == 9]


def sentences(texts):
    """Split every story at . ! or ? followed by a space."""
    return [s for t in texts for s in re.split(r"(?<=[.!?])\s+", t) if len(s.split()) > 1]


train_sent, val_sent, test_sent = sentences(train_stories), sentences(val_stories), sentences(test_stories)
print(len(stories), "stories,", sum(len(f.split()) for f in stories), "words")
print(len(train_stories), len(val_stories), len(test_stories), "stories;",
      len(train_sent), len(val_sent), len(test_sent), "sentences (train, validation, test)")
print(train_sent[:3])

## 2. A number for every word: the 3,000 most frequent training words
Every other word becomes [UNK] (index 1). Index 0 is padding.

In [ ]:
VOCAB = 3000
vectorizer = keras.layers.TextVectorization(max_tokens=VOCAB)   # lower-case, strip punctuation, split on spaces
vectorizer.adapt(train_sent)
vocab = vectorizer.get_vocabulary()                  # index 0 = padding '', index 1 = [UNK]
V = len(vocab)
full = keras.layers.TextVectorization()           # same, without the cap, only to count the words
full.adapt(train_sent)
n_distinct = len(full.get_vocabulary()) - 2
print("distinct words in the training stories:", n_distinct, "  vocabulary size (with '' and [UNK]):", V)
print(vocab[:12])


def to_ids(s):
    return [int(i) for i in vectorizer([s]).numpy()[0] if i != 0]


print(train_sent[0], "->", to_ids(train_sent[0]))

## 3. Input-output pairs: every prefix of a sentence predicts its next word
A pair whose next word is [UNK] is dropped: no model can name a word it has never seen.

In [ ]:
def ngrams(sents):
    seqs, unk = [], 0
    for s in sents:
        ids = to_ids(s)
        for i in range(1, len(ids)):
            if ids[i] == 1:
                unk += 1
                continue
            seqs.append(ids[:i + 1])        # first i words as input, word i+1 as output
    return seqs, unk


(train_seqs, unk_tr), (val_seqs, unk_va), (test_seqs, unk_te) = ngrams(train_sent), ngrams(val_sent), ngrams(test_sent)
for s in train_seqs[:4]:
    print(s[:-1], "->", s[-1], "   ", " ".join(vocab[i] for i in s[:-1]), "->", vocab[s[-1]])
max_len = 51          # at most the last 50 words before the target (a few sentences are much longer)
print("pairs:", len(train_seqs), "training,", len(val_seqs), "validation,", len(test_seqs), "test")
print("dropped because the next word is [UNK]:", unk_tr, unk_va, unk_te,
      f"(share of test next words: {unk_te / (unk_te + len(test_seqs)):.3f})")
lengths = np.array([len(s) for s in train_seqs])
print("longest sequence:", lengths.max(), " share of training pairs with more than 50 input words:", round(float(np.mean(lengths > 51)), 3))

In [ ]:
# Zeros in front, so the last word of the input is always just before the output
def split(seqs):
    pad = keras.utils.pad_sequences(seqs, maxlen=max_len, padding="pre", truncating="pre")
    return pad[:, :-1], pad[:, -1]


X, y = split(train_seqs)
X_val, y_val = split(val_seqs)
X_test, y_test = split(test_seqs)
print("first padded row ends:", X[0][-6:], y[0], " X shape", X.shape, " y shape", y.shape)

In [ ]:
# Classification over the vocabulary: one-hot targets, one column per word
Y = keras.utils.to_categorical(y, num_classes=V)
Y_val = keras.utils.to_categorical(y_val, num_classes=V)
print("Y shape:", Y.shape, " the first target is word", y[0], "=", repr(vocab[y[0]]))

## 4. Two simple baselines, scored on the test pairs
- always guess the most common next word of the training pairs;
- bigram: guess the word that most often follows the previous word in the training pairs
  (the most common word if the previous word was never followed by a known word).

In [ ]:
most_common = Counter(y.tolist()).most_common(1)[0][0]
follow = defaultdict(Counter)
for s in train_seqs:
    follow[s[-2]][s[-1]] += 1
bigram = {w: c.most_common(1)[0][0] for w, c in follow.items()}
bigram_pred = np.array([bigram.get(s[-2], most_common) for s in test_seqs])
base = {"always the most common word": float(np.mean(y_test == most_common)),
        "bigram (previous word only)": float(np.mean(bigram_pred == y_test))}
print("most common next word:", repr(vocab[most_common]))
for k, v in base.items():
    print(f"{k:30} test accuracy {v:.3f}")

## 5. Embedding -> LSTM -> Dense softmax, trained with early stopping
Up to 30 epochs; training stops when the validation loss has not improved for 3 epochs, and the weights of the
best epoch are kept. Five seeds; the test accuracy is their mean.

In [ ]:
def build():
    model = keras.Sequential([
        keras.Input(shape=(max_len - 1,)),
        keras.layers.Embedding(V, 100),
        keras.layers.LSTM(150),
        keras.layers.Dense(V, activation="softmax")])
    model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model


build().summary()
counts = [l.count_params() for l in build().layers]
print("parameters per layer:", counts, "total", sum(counts))
assert counts[1] == 4 * ((150 + 100) * 150 + 150)

In [ ]:
SEEDS, rows, hists, models = (0, 1, 2, 3, 4), [], [], {}
for seed in SEEDS:
    keras.utils.set_random_seed(seed)
    model = build()
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    hist = model.fit(X, Y, epochs=30, batch_size=64, validation_data=(X_val, Y_val), callbacks=[stop], verbose=0)
    h = pd.DataFrame(hist.history)
    h.insert(0, "epoch", np.arange(1, len(h) + 1))
    h.insert(0, "seed", seed)
    hists.append(h)
    best = int(h.epoch[h.val_loss.idxmin()])
    p = model.predict(X_test, verbose=0)
    p[:, :2] = 0                                   # never predict padding or [UNK] (no test target is either)
    acc = float(np.mean(p.argmax(1) == y_test))
    rows.append(dict(seed=seed, best_epoch=best, epochs_run=len(h), test_accuracy=acc,
                     train_accuracy_best=float(h.accuracy[best - 1]), val_accuracy_best=float(h.val_accuracy[best - 1])))
    models[seed] = model
    print(f"seed {seed}: best epoch {best} of {len(h)}, test accuracy {acc:.3f}", flush=True)
hist_all = pd.concat(hists)
hist_all.round(4).to_csv(DATA / "history.csv", index=False)
runs = pd.DataFrame(rows)
runs.round(4).to_csv(DATA / "runs.csv", index=False)
print(runs.round(3).to_string(index=False))
print("LSTM test accuracy: mean", round(runs.test_accuracy.mean(), 3), "std", round(runs.test_accuracy.std(), 3),
      "min", round(runs.test_accuracy.min(), 3), "max", round(runs.test_accuracy.max(), 3))
summary = pd.DataFrame([dict(method=k, test_accuracy=v) for k, v in base.items()] +
                       [dict(method="LSTM (mean of 5 seeds)", test_accuracy=runs.test_accuracy.mean())])
summary["test_accuracy_std"] = [0.0, 0.0, runs.test_accuracy.std()]
summary.round(4).to_csv(DATA / "baselines.csv", index=False)
print(summary.round(3).to_string(index=False))

## 6. Does the LSTM use more than the previous word?
Test pairs split by whether the bigram baseline gets them right.

In [ ]:
model = models[0]
p = model.predict(X_test, verbose=0)
p[:, :2] = 0
lstm_pred = p.argmax(1)
ok_bi = bigram_pred == y_test
print("LSTM accuracy where the bigram is right:", round(float(np.mean(lstm_pred[ok_bi] == y_test[ok_bi])), 3),
      " where the bigram is wrong:", round(float(np.mean(lstm_pred[~ok_bi] == y_test[~ok_bi])), 3))

## 7. Training on past the best epoch (seed 0)

In [ ]:
h0 = hist_all[hist_all.seed == 0]
print(h0.round(3).to_string(index=False))

## 8. Predicting the next word, then the next 10 (model of seed 0)

In [ ]:
def next_word(text):
    ids = keras.utils.pad_sequences([to_ids(text)], maxlen=max_len - 1, padding="pre", truncating="pre")
    p = model.predict(ids, verbose=0)[0]          # one probability per word of the vocabulary
    p[:2] = 0                                     # skip padding and [UNK]
    return vocab[int(np.argmax(p))], float(p.max())


def generate(text, n=10):
    for _ in range(n):
        text = text + " " + next_word(text)[0]
    return text


prompts = ["a certain king had a beautiful garden and in the", "the king", "the wolf", "once upon a time", "the old woman said"]
print(prompts[0], "->", next_word(prompts[0]))
rows = []
for p in prompts:
    out = generate(p, 10)
    rows.append(dict(prompt=p, generated=out[len(p) + 1:]))
    print(f"{p} -> {out[len(p) + 1:]}")
pd.DataFrame(rows).to_csv(DATA / "generated.csv", index=False)

In [ ]:
# The real continuation of the first prompt, for comparison
print(train_sent[0])

Typing a sentence from a test story word by word: after each prefix, the model's five most likely next words (seed 0). The sentence is the first test sentence of 8 to 14 known words.

In [ ]:
typed = next(s for s in test_sent if 8 <= len(to_ids(s)) <= 14 and 1 not in to_ids(s))
ids = to_ids(typed)
rows = []
for k in range(1, len(ids)):
    x = keras.utils.pad_sequences([ids[:k]], maxlen=max_len - 1, padding="pre", truncating="pre")
    p = model.predict(x, verbose=0)[0]
    p[:2] = 0                                     # skip padding and [UNK], as in next_word
    rank = int((p > p[ids[k]]).sum()) + 1         # where the real next word sits among the guesses
    for r, j in enumerate(np.argsort(p)[::-1][:5], start=1):
        rows.append(dict(step=k, prefix=" ".join(vocab[i] for i in ids[:k]), next_word=vocab[ids[k]],
                         next_p=float(p[ids[k]]), next_rank=rank, rank=r, word=vocab[j], p=float(p[j])))
typing = pd.DataFrame(rows)
typing.round(4).to_csv(DATA / "typing.csv", index=False)
print(typed)
print(typing[typing["rank"] == 1][["step", "prefix", "word", "p", "next_word", "next_p", "next_rank"]].round(3).to_string(index=False))